# SmartPlant AI — 50-class research training
**Do not run training before obtaining all 50 correctly labeled classes.** This notebook uses the isolated development pipeline (not current production `main`). Dataset sources, licenses, original photographs and plant/farm groups must be reviewed independently.

In [ ]:
%cd /content
!git clone --depth 1 --branch feature/smartplant-expanded-cnn-50 https://github.com/dapperdude404/tinyguard-ai.git
%cd /content/tinyguard-ai
!python -m pip install -q gdown pillow numpy scikit-learn

## Mount your own Google Drive
Create `MyDrive/SmartPlant50/data/expanded/`, containing a class plan, `train/`, `val/`, `test/`, and a provenance CSV. **Do not upload raw images to public GitHub.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/SmartPlant50/data/expanded')
print('Dataset folder exists:', DATA.is_dir())

## Verify original checkpoint and 50-class label plan
The trusted original checkpoint is SHA-256 pinned. The plan preserves all original 29 class positions. Only run if your dataset has been independently reviewed.

In [ ]:
!python -c "from smartplant.model_download import ensure_cnn_checkpoint; print(ensure_cnn_checkpoint('models/smartplant_mobilenetv3.pt'))"
!python scripts/prepare_expanded_labels.py --checkpoint models/smartplant_mobilenetv3.pt --output data/expanded/class_plan.json
print('Candidate class plan created (not proof of labeled images)')

## Dataset readiness gate
This gate intentionally stops training when any class is missing. `groups.csv` must assign each image to a source plant/farm and prevent leakage across splits.

In [ ]:
assert DATA.is_dir(), 'Missing Drive data folder'
assert (DATA / 'groups.csv').is_file(), 'Missing groups.csv: obtain independent source-plant/farm IDs before training'
for split in ('train','val','test'):
    for index in range(50):
        assert (DATA / split / f'c{index:03d}').is_dir(), f'Missing {split}/c{index:03d}'
print('All class directories and provenance file located; running actual integrity audit next')

In [ ]:
!python scripts/audit_expanded_dataset.py --dataset "{DATA}" --plan data/expanded/class_plan.json --groups-csv "{DATA / 'groups.csv'}"

## Fine-tune ONLY after the audit passes
Validation macro-F1 determines early stopping. Test metrics are computed once after selection. The resulting accuracy is not evidence of independent field reliability without a proper external evaluation.

In [ ]:
!python scripts/train_expanded_cnn.py --dataset "{DATA}" --plan data/expanded/class_plan.json --groups-csv "{DATA / 'groups.csv'}" --base-checkpoint models/smartplant_mobilenetv3.pt --output "{DATA.parent / 'smartplant_50.pt'}" --seed 42 --epochs 25 --batch-size 32 --patience 6 --lr 0.0001

## Evaluate before any Android/Web model replacement
Review `.metrics.json` and `.confusion.csv`, test unseen field photos, non-leaf inputs, crop mismatches, and verify license/provenance. Only then generate a SHA-pinned ONNX model on a separate testing branch. **Do not merge or deploy automatically.**